In [1]:
import os
import re
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import pandas as pd


## Mateo CIFAR10

In [ ]:
import os
import torch
import numpy as np

root = os.path.join("datasets", "CIFAR10")

for name in ["c_train.pt", "c_test.pt",
             "c_train_percentile_threshold_bool.pt",
             "c_test_percentile_threshold_bool.pt",
             "c_train_zero_shot_bool.pt",
             "c_test_zero_shot_bool.pt",
             "cifar10_concepts_train.pt"]:
    t = torch.load(os.path.join(root, name), map_location="cpu")
    print(name, type(t).__name__,
          getattr(t, "shape", len(t) if hasattr(t, "__len__") else None),
          getattr(t, "dtype", None))

train_idxs = np.load(os.path.join(root, "train_idxs.npy"))
val_idxs = np.load(os.path.join(root, "val_idxs.npy"))
print("split sizes:", train_idxs.shape, val_idxs.shape, "overlap:",
      len(np.intersect1d(train_idxs, val_idxs)))

c_tr = torch.load(os.path.join(root, "c_train.pt"), map_location="cpu").float()
c_te = torch.load(os.path.join(root, "c_test.pt"), map_location="cpu").float()
b_tr = torch.load(os.path.join(root, "c_train_percentile_threshold_bool.pt"), map_location="cpu")
z_tr = torch.load(os.path.join(root, "c_train_zero_shot_bool.pt"), map_location="cpu")

print("score range:", c_tr.min().item(), c_tr.max().item())

for tag, b in [("percentile", b_tr), ("zero-shot", z_tr)]:
    p = b.float().mean(0)
    print(f"{tag} prevalence  min={p.min():.4f} mean={p.mean():.4f} max={p.max():.4f}")

# where were the thresholds estimated?
for tag, ref in [("train only", c_tr), ("train+test", torch.cat([c_tr, c_te], 0))]:
    thr = ref.median(0).values
    print(f"{tag}: agreement with shipped labels =",
          ((c_tr > thr) == b_tr).float().mean().item())

In [ ]:
for name in ["cifar10_train_clip_ViT-B16.pt", "cifar10_train_resnet50_layer4.pt",
             "cifar10_val_resnet50_layer4.pt", "cifar10_test_clip_ViT-B16.pt"]:
    t = torch.load(os.path.join(root, name), map_location="cpu")
    print(name, getattr(t, "shape", None), getattr(t, "dtype", None))

# Intervention curves for CIFAR-10

In [4]:
def extract_info_from_intervention_log(model_dir, model_type="scbm", dataset="cifar10", return_c_auroc=False):
    experiment_path = os.path.join("experiments", model_type, dataset)
    concepts_intervened_list = []
    y_accuracies_list = []
    c_accuracies_list = []
    c_auroc_list = []

    with open(os.path.join(experiment_path, model_dir, "intervention_log.txt"), "r") as f:
        lines = f.readlines()
        for line in lines:
            if "Intervention on" in line and "y_accuracy" in line and "c_accuracy" in line:
                m = re.search(r"Intervention on (\d+) concepts", line)
                if not m:
                    continue
                concepts_intervened = int(m.group(1))

                y_m = re.search(r"y_accuracy:\s*([0-9.]+)", line)
                c_m = re.search(r"c_accuracy:\s*([0-9.]+)", line)
                c_auroc_m = re.search(r"c_AUROC:\s*([0-9.]+)", line)

                y_accuracy = float(y_m.group(1)) * 100 if y_m else None
                c_accuracy = float(c_m.group(1)) * 100 if c_m else None
                c_auroc = float(c_auroc_m.group(1)) * 100 if c_auroc_m else None

                concepts_intervened_list.append(concepts_intervened)
                y_accuracies_list.append(y_accuracy)
                c_accuracies_list.append(c_accuracy)
                c_auroc_list.append(c_auroc)
    if return_c_auroc:
        return concepts_intervened_list, y_accuracies_list, c_accuracies_list, c_auroc_list
    return concepts_intervened_list, y_accuracies_list, c_accuracies_list


In [2]:
model_dir_scbm = "cifar10_scbm_2026-09-03_19-11-53_7001f"
model_dir_scbm_res_20 = "cifar10_scbm_res_20_2026-09-03_18-01-41_e9588"
model_dir_scbm_res_20_l_int_ext = (
    "cifar10_scbm_res_20_rtx_L_int_extension_loss_weight_1_2026-09-03_19-14-43_004be"
)

(concepts_intervened_list_scbm,
 y_accuracies_list_scbm,
 c_accuracies_list_scbm,
 c_auroc_list_scbm) = extract_info_from_intervention_log(
    model_dir_scbm, model_type="scbm", dataset="cifar10", return_c_auroc=True
)

(concepts_intervened_list_scbm_res_20,
 y_accuracies_list_scbm_res_20,
 c_accuracies_list_scbm_res_20,
 c_auroc_list_scbm_res_20) = extract_info_from_intervention_log(
    model_dir_scbm_res_20, model_type="scbm_residual", dataset="cifar10", return_c_auroc=True
)

(concepts_intervened_list_scbm_res_20_l_int_ext,
 y_accuracies_list_scbm_res_20_l_int_ext,
 c_accuracies_list_scbm_res_20_l_int_ext,
 c_auroc_list_scbm_res_20_l_int_ext) = extract_info_from_intervention_log(
    model_dir_scbm_res_20_l_int_ext, model_type="scbm_residual", dataset="cifar10", return_c_auroc=True
)

plt.style.use("seaborn-v0_8-whitegrid")
fig, ax = plt.subplots(figsize=(11, 6))

ax.plot(
    concepts_intervened_list_scbm,
    y_accuracies_list_scbm,
    linewidth=2.2,
    markersize=6,
    color="#1f77b4",
    label="SCBM",
)

ax.plot(
    concepts_intervened_list_scbm_res_20,
    y_accuracies_list_scbm_res_20,
    linewidth=2.2,
    markersize=6,
    color="#ff7f0e",
    linestyle="--",
    label="SCBM + residual (20 residuals)",
)

ax.plot(
    concepts_intervened_list_scbm_res_20_l_int_ext,
    y_accuracies_list_scbm_res_20_l_int_ext,
    linewidth=2.2,
    markersize=6,
    color="#2ca02c",
    linestyle="-.",
    label="SCBM + residual (20 residuals) + extended $L_{int}$",
)

ax.set_title(
    "Task Accuracy vs Number of Concepts Intervened on (CIFAR-10)",
    fontsize=14,
    pad=12,
)
ax.set_xlabel("Number of concepts intervened on", fontsize=11)
ax.set_ylabel("Task accuracy (%)", fontsize=11)

max_len = max(
    len(concepts_intervened_list_scbm),
    len(concepts_intervened_list_scbm_res_20),
    len(concepts_intervened_list_scbm_res_20_l_int_ext),
)
ax.xaxis.set_major_locator(MaxNLocator(integer=True, prune=None, nbins=min(20, max_len)))

y_min = min(
    min(y_accuracies_list_scbm),
    min(y_accuracies_list_scbm_res_20),
    min(y_accuracies_list_scbm_res_20_l_int_ext),
)
y_max = max(
    max(y_accuracies_list_scbm),
    max(y_accuracies_list_scbm_res_20),
    max(y_accuracies_list_scbm_res_20_l_int_ext),
)
pad = max((y_max - y_min) * 0.1, 0.5)
ax.set_ylim(y_min - pad, y_max + pad)

ax.grid(True, which="major", linestyle="--", alpha=0.45)
ax.legend(loc="best", frameon=True)
fig.tight_layout()
plt.show()

NameError: name 'extract_info_from_intervention_log' is not defined